[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/19_kubernetes_helm/19_proyecto_kubernetes_helm.ipynb)

# Proyecto 19 · Despliega el stack y SupportOps en Kubernetes (con Helm) y verifica el flujo de telemetría

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 2 h 30 min |
| **Servicios de Compose** | Ninguno. Necesita `helm` y, para R1/R4/R7, Docker (contenedores efímeros y un API server k3s). Sin ellos, esas comprobaciones se omiten y el resto puntúa |
| **RAM aproximada** | ≈ 1-2 GB durante la rúbrica |
| **Prerrequisitos** | Lección `19_kubernetes_helm`; proyecto 18 (la política de muestreo) y módulo 05 |
| **Conexión** | Despliega como código la plataforma completa de los módulos 05-18 (Collector de dos capas con la política de muestreo, Tempo, Loki, Prometheus, Grafana) y el agente SupportOps; es el «Despliegue» del capstone |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Producción pasa de `docker compose` a Kubernetes y el equipo de seguridad exige: **requests/limits, usuario no-root, sistema de ficheros de solo lectura**, validación automática en CI de todo lo que se despliegue,
autoscaling del gateway **sin romper el tail sampling**, y la opción de gestionar el Collector con el Operator. Tú escribes las piezas de código que lo comprueban y generan; la rúbrica las ejecuta contra los charts, los binarios y un API server reales.

## Datos

Los charts y values del módulo (`k8s/`), 7 releases ya renderizados en `REND` y el flujo medido en `REC["flow"]`. Sin datos externos.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución) |
|---|---|---|---|
| R1 | `gateway_overrides` | `helm template` con tus capas: `decision_wait`, `num_traces`, baseline, nº de réplicas; `kubeconform` y `otelcol validate` | exacto; 0 inválidos |
| R2 | `memory_budget` | suma de requests de memoria de todos los releases frente a un nodo | ≈ 1,4 GiB total ≤ 50 % de 8 GiB; Tempo solo no cabe en 200 MiB |
| R3 | `security_findings` | hallazgos por release (sin `limits.memory`, sin `runAsNonRoot`, sin `readOnlyRootFilesystem`) | 0 en `supportops`; mismos que la referencia en el resto |
| R4 | `validate_rendered` | las configuraciones renderizadas pasan por **su** binario | 4/4 (con Docker) |
| R5 | `needs_affinity` | qué políticas de tail sampling exigen que la traza llegue entera a una réplica | exacto |
| R6 | `desired_replicas` | algoritmo del HPA | 8/8 casos idénticos al de Kubernetes |
| R7 | `operator_cr` | CR `OpenTelemetryCollector` v1beta1 derivado de tus values; el API server real lo valida | estructura + `--dry-run=server` OK |
| R8 | `verify_flow` | criterios de aceptación del flujo extremo a extremo | aprueba el medido y suspende uno roto |

> 📏 Los umbrales salen de la solución con los charts verificados (Collector 0.176.0, Tempo 3.2.0, Loki 18.16.0, Grafana 14.0.0). Si usas otras versiones de chart, las cifras de R2/R3 pueden cambiar.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, copy, gzip, json, math, os, pathlib, subprocess, time
import yaml
from IPython.display import Markdown, display
from supportops import k8s as K
from supportops import sampling as S
K8S = ROOT / "19_kubernetes_helm" / "k8s"
VALUES = K8S / "values"
REC = json.load(gzip.open(ROOT / "19_kubernetes_helm" / "recorded" / "capture_seed42.json.gz", "rt", encoding="utf-8"))   # medido con helm, k3s y los binarios reales (modo degradado)
TOOLS = K.tool_versions()
HELM = TOOLS["helm"] != "—"
DOCKER = S.docker_ok()
CHARTS_OK = HELM and all(K.chart_path(k, pull=True) for k in K.CHARTS)       # los 5 charts oficiales + los 2 propios, en local
LIVE_RENDER = HELM and CHARTS_OK
print(f"helm: {TOOLS['helm']} · docker: {DOCKER} · charts disponibles en local: {CHARTS_OK}")
if not LIVE_RENDER:
    print("→ modo degradado: lo que necesita helm o los charts usa los resultados pregrabados (mismo código, medido al construir el módulo).")

PLAN = [("otel-gateway", "otel-collector", ["otel-gateway.yaml"], "observability"), ("otel-agent", "otel-collector", ["otel-agent.yaml"], "observability"),
        ("tempo", "tempo", ["tempo.yaml"], "observability"), ("loki", "loki", ["loki.yaml"], "observability"), ("grafana", "grafana", ["grafana.yaml"], "observability"),
        ("prometheus", "prom-lab", [], "observability"), ("supportops", "supportops", ["supportops-lab.yaml"], "supportops")]

## Paso 0 · Los releases renderizados (te lo damos hecho)

`REND` contiene los manifiestos de los 7 releases (o, sin helm, los datos pregrabados reducidos). `API_LAB` es un API server k3s real si hay Docker (se usa en R7; se destruye al final de la rúbrica).

In [ ]:
if LIVE_RENDER:
    REND = {rel: K.render(rel, ch, [VALUES / v for v in vals], namespace=ns) for rel, ch, vals, ns in PLAN}
    print({k: len(v) for k, v in REND.items()}, "manifiestos")
else:
    REND = {}
    print("sin helm/charts: R1-R4 se omiten; el resto de pasos sigue funcionando")
API_LAB = None

## Paso 1 · El gateway por capas de values (TODO)

Escribe `gateway_overrides(decision_wait, num_traces, baseline_pct, replicas)` → un dict de **values** (solo lo que cambia) que, aplicado sobre `k8s/values/otel-gateway.yaml`, fija `decision_wait`, `num_traces`, el `sampling_percentage` de la política `baseline` y `autoscaling.minReplicas`. ⚠️ Recuerda la regla de fusión de la lección §2: **las listas se reemplazan enteras** (`policies` incluida: repite las cuatro, con los mismos nombres y en el mismo orden: `errors`, `slow`, `costly`, `baseline`).

<details>
<summary>🪜 Pista 1</summary>

Las capas de values se aplican de izquierda a derecha; la tuya gana. Solo necesitas escribir las claves que cambian, pero `policies` es una lista: se reemplaza completa.

</details>

<details>
<summary>🪜 Pista 2</summary>

`{"autoscaling": {"minReplicas": replicas}, "config": {"processors": {"tail_sampling": {"decision_wait": ..., "num_traces": ..., "policies": [ {...errors}, {...slow}, {...costly OTTL}, {...baseline} ]}}}}`; copia las políticas del módulo 18 (el coste con `ottl_condition`, no `numeric_attribute`).

</details>

In [ ]:
def gateway_overrides(decision_wait, num_traces, baseline_pct, replicas):
    # TODO: {"autoscaling": {"minReplicas": ...}, "config": {"processors": {"tail_sampling": {...}}}}
    raise NotImplementedError("TODO 1")

try:
    if LIVE_RENDER:
        ms = K.render("otel-gateway", "otel-collector", [VALUES / "otel-gateway.yaml", gateway_overrides("20s", 30000, 5, 3)])
        print(yaml.safe_load(K.configmap_text(ms, "otel-gateway", "relay"))["processors"]["tail_sampling"]["policies"][-1])
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 2 · El presupuesto de memoria (TODO)

Escribe `memory_budget(manifests, node_mem_mib, headroom=0.5)` → `(cabe, total_mib)`: suma las `requests` de memoria de los manifiestos (usa `K.resource_requests`) y comprueba que no superan `headroom` × la memoria del nodo (dejamos la mitad libre para el sistema y los picos).

<details>
<summary>🪜 Pista 1</summary>

`K.resource_requests(manifiestos)` ya devuelve `{"cpu_m", "memory_mib"}` con las réplicas contadas.

</details>

<details>
<summary>🪜 Pista 2</summary>

`total = K.resource_requests(manifests)["memory_mib"]; return total <= node_mem_mib * headroom, total`.

</details>

In [ ]:
def memory_budget(manifests, node_mem_mib, headroom=0.5):
    # TODO
    raise NotImplementedError("TODO 2")

try:
    if LIVE_RENDER:
        print(memory_budget(sum(REND.values(), []), 8192, 0.5), memory_budget(REND["tempo"], 200, 1.0))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 3 · Seguridad (TODO)

Escribe `security_findings(manifests)` → lista de textos (uno por hallazgo) para cada contenedor de Deployment/StatefulSet/DaemonSet que **(a)** no tenga `resources.limits.memory`, **(b)** no tenga `runAsNonRoot: true` (a nivel de pod **o** de contenedor) o **(c)** no tenga `securityContext.readOnlyRootFilesystem: true`. Formato: `"Deployment/nombre/contenedor: motivo"`.

<details>
<summary>🪜 Pista 1</summary>

Los manifiestos son dicts de Python (salida de `yaml.safe_load`): `m["kind"]`, `m["metadata"]["name"]`, `m["spec"]["template"]["spec"]["containers"]`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Usa `(c.get("securityContext") or {})` y `(pod.get("securityContext") or {})` para evitar `KeyError` cuando falta el bloque.

</details>

In [ ]:
def security_findings(manifests):
    # TODO: recorre manifests; pod = m["spec"]["template"]["spec"]; contenedores en pod["containers"]
    raise NotImplementedError("TODO 3")

try:
    if LIVE_RENDER:
        for k, v in REND.items():
            print(k, len(security_findings(v)))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 4 · Valida lo renderizado con los binarios (TODO)

Escribe `validate_rendered(rendered)` → `{"otel-gateway": bool, "otel-agent": bool, "tempo": bool, "loki": bool}`: extrae cada ConfigMap con `K.configmap_text` (claves `relay`, `relay`, `tempo.yaml`, `config.yaml`) y valida con `K.validate_with_binary(kind, texto)[0]` (`kind` ∈ `collector|tempo|loki`).

<details>
<summary>🪜 Pista 1</summary>

`K.configmap_text(manifiestos, nombre_del_configmap, clave)`; el ConfigMap del gateway se llama `otel-gateway` y su clave es `relay`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`{"otel-gateway": K.validate_with_binary("collector", K.configmap_text(rendered["otel-gateway"], "otel-gateway", "relay"))[0], ...}`.

</details>

In [ ]:
def validate_rendered(rendered):
    # TODO
    raise NotImplementedError("TODO 4")

try:
    if LIVE_RENDER and DOCKER:
        print(validate_rendered(REND))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 5 · ¿Cuándo hace falta afinidad? (TODO)

Escribe `needs_affinity(policies)` → `True` si **alguna** política de la lista (dicts con `type`) depende de ver la traza **entera**. Piensa en la lección §9: `probabilistic` decide solo con el `traceID` (todas las réplicas deciden igual), mientras que las que miran spans, estado, duración o número de spans necesitan toda la traza en una réplica.

<details>
<summary>🪜 Pista 1</summary>

Una política «por traceID» da el mismo veredicto en cualquier réplica; una política «por contenido» solo es correcta si la réplica tiene todos los spans.

</details>

<details>
<summary>🪜 Pista 2</summary>

`any(p["type"] != "probabilistic" for p in policies)` sirve como primera aproximación; la referencia usa la lista explícita de tipos.

</details>

In [ ]:
def needs_affinity(policies):
    # TODO: tipos que necesitan la traza completa: latency, status_code, span_count, numeric_attribute, string_attribute, boolean_attribute, ottl_condition, trace_state, and, composite
    raise NotImplementedError("TODO 5")

try:
    print(needs_affinity([{"type": "probabilistic"}]), needs_affinity([{"type": "status_code"}, {"type": "probabilistic"}]))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 6 · El HPA (TODO)

Escribe `desired_replicas(current, metric, target, min_r, max_r, tol=0.1)` con el algoritmo del HorizontalPodAutoscaler: `ceil(current × métrica/objetivo)`, sin cambios si `|métrica/objetivo − 1| ≤ tol`, acotado a `[min_r, max_r]`.

<details>
<summary>🪜 Pista 1</summary>

La razón métrica/objetivo decide todo: dentro de la tolerancia no hay cambio.

</details>

<details>
<summary>🪜 Pista 2</summary>

`ratio = metric/target; d = current if abs(ratio-1) <= tol else math.ceil(current*ratio); return max(min_r, min(max_r, d))`.

</details>

In [ ]:
def desired_replicas(current, metric, target, min_r, max_r, tol=0.1):
    # TODO
    raise NotImplementedError("TODO 6")

try:
    print([desired_replicas(3, 98, 70, 2, 10), desired_replicas(3, 74, 70, 2, 10), desired_replicas(6, 700, 70, 2, 8)])
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 7 · Del chart al Operator (TODO)

Escribe `operator_cr(values, name="gateway", namespace="observability")` → el dict de un CR `OpenTelemetryCollector` (`opentelemetry.io/v1beta1`) derivado de los **values** del gateway: `mode`, `image` (`repositorio:tag`), `replicas`, `resources`, `config` (receivers/processors/exporters/connectors/service; en v1beta1 es un **mapa**), `autoscaler` (`minReplicas`, `maxReplicas`, `targetCPUUtilization`) si `autoscaling.enabled`, y `podDisruptionBudget` si está habilitado.

<details>
<summary>🪜 Pista 1</summary>

El modelo es `k8s/operator/collector-gateway.yaml`. Los nombres de los campos del CR no son los de los values (`targetCPUUtilizationPercentage` → `targetCPUUtilization`).

</details>

<details>
<summary>🪜 Pista 2</summary>

`spec = {"mode": ..., "image": f"{repo}:{tag}", "replicas": values["replicaCount"], "resources": ..., "config": {k: values["config"][k] for k in (...)}}`; la config de los values contiene claves con `None` (borrados de Helm): ¡fíltralas por las cinco claves de la lista!

</details>

In [ ]:
def operator_cr(values, name="gateway", namespace="observability"):
    # TODO: ver k8s/operator/collector-gateway.yaml como modelo
    raise NotImplementedError("TODO 7")

try:
    vals = yaml.safe_load((VALUES / "otel-gateway.yaml").read_text())
    print(list(operator_cr(vals)["spec"]))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 8 · Criterios de aceptación del flujo (TODO)

Escribe `verify_flow(flow)` → dict `{criterio: bool}` con cinco comprobaciones sobre el resultado del flujo (`REC["flow"]`: `tickets`, `traces_in_tempo`, `loki_lines`, `prom_genai_series`, `span_metrics_calls`, `gateway_up`): trazas conservadas entre el 10 % y el 40 % de los tickets, hay logs, hay métricas `gen_ai`, hay `span_metrics` con llamadas y todas las réplicas del gateway están `up` (=1).

<details>
<summary>🪜 Pista 1</summary>

Un criterio de aceptación es una proposición booleana sobre las métricas medidas; el 10-40 % sale de la política (≈ 10 % base + lo interesante).

</details>

<details>
<summary>🪜 Pista 2</summary>

`frac = flow["traces_in_tempo"]/flow["tickets"]`; `{"trazas...": 0.10 <= frac <= 0.40, "hay logs...": flow["loki_lines"] > 0, ...}`.

</details>

In [ ]:
def verify_flow(flow):
    # TODO
    raise NotImplementedError("TODO 8")

try:
    print(verify_flow(REC["flow"]))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Rúbrica · ejecútala cuando quieras

In [ ]:
EXPECTED_FINDINGS = REC["expected_findings"]
def run_rubric(live=LIVE_RENDER, docker=DOCKER):
    R = {}
    def chk(name, fn, needs=True):
        if not needs:
            R[name] = None; return
        try:
            R[name] = bool(fn())
        except NotImplementedError:
            R[name] = None
        except Exception as e:
            R[name] = False; print(f"  {name}: {type(e).__name__}: {str(e)[:300]}")
    def r1():
        ov = gateway_overrides("20s", 30000, 5, 3)
        ms = K.render("otel-gateway", "otel-collector", [VALUES / "otel-gateway.yaml", ov])
        cfg = yaml.safe_load(K.configmap_text(ms, "otel-gateway", "relay"))
        ts = cfg["processors"]["tail_sampling"]
        hpa = next(m for m in ms if m["kind"] == "HorizontalPodAutoscaler")
        pols = {p["name"]: p for p in ts["policies"]}
        return (ts["decision_wait"] == "20s" and ts["num_traces"] == 30000 and pols["baseline"]["probabilistic"]["sampling_percentage"] == 5 and list(pols) == ["errors", "slow", "costly", "baseline"]
                and hpa["spec"]["minReplicas"] == 3 and K.kubeconform(ms)["invalid"] == 0 and (not docker or K.validate_with_binary("collector", K.configmap_text(ms, "otel-gateway", "relay"))[0]))
    chk("R1 gateway_overrides: values por capas → decision_wait, num_traces, baseline y réplicas correctos; kubeconform y otelcol validate OK", r1, live)
    def r2():
        tot = {k: K.resource_requests(v)["memory_mib"] for k, v in REND.items()}
        ok, total = memory_budget(sum(REND.values(), []), 8192, 0.5)
        ok2, _ = memory_budget(REND["tempo"], 200, 1.0)
        return ok and abs(total - sum(tot.values())) < 1e-6 and ok2 is False
    chk("R2 memory_budget: suma de requests de todos los releases (≈ 1,4 GiB) cabe en el 50 % de 8 GiB y Tempo solo NO cabe en 200 MiB", r2, live)
    def r3():
        counts = {k: len(security_findings(v)) for k, v in REND.items()}
        return counts == EXPECTED_FINDINGS and counts["supportops"] == 0
    chk("R3 security_findings: 0 hallazgos en el chart propio y los mismos que la referencia en el resto de releases", r3, live)
    chk("R4 validate_rendered: gateway, agente, Tempo y Loki válidos según SUS binarios (Docker)", lambda: all(validate_rendered(REND).values()), live and docker)
    chk("R5 needs_affinity: errors/latency exigen afinidad por traceID; solo probabilistic, no", lambda: needs_affinity([{"type": "status_code"}, {"type": "probabilistic"}]) and needs_affinity([{"type": "latency"}])
        and not needs_affinity([{"type": "probabilistic"}]))
    chk("R6 desired_replicas: coincide con K.hpa_desired en 8 casos (tolerancia del 10 %, mínimos y máximos)", lambda: all(desired_replicas(*a) == K.hpa_desired(*a) for a in
        [(3, 98, 70, 2, 10), (3, 74, 70, 2, 10), (4, 20, 70, 2, 10), (6, 700, 70, 2, 8), (2, 70, 70, 1, 5), (5, 40, 70, 3, 9), (1, 1000, 70, 1, 4), (10, 66, 70, 2, 12)]))
    def r7():
        vals = yaml.safe_load((VALUES / "otel-gateway.yaml").read_text())
        cr = operator_cr(vals)
        ok = (cr["apiVersion"] == "opentelemetry.io/v1beta1" and cr["kind"] == "OpenTelemetryCollector" and cr["spec"]["autoscaler"]["maxReplicas"] == 6 and cr["spec"]["podDisruptionBudget"]["minAvailable"] == 1
              and "tail_sampling" in cr["spec"]["config"]["processors"] and isinstance(cr["spec"]["config"], dict))
        if not (live and docker):
            return ok
        cr["spec"]["config"]["service"]["pipelines"]["traces/in"]  # existe
        r = API_LAB.apply([cr], dry_run=True)
        return ok and r.returncode == 0
    chk("R7 operator_cr: CR v1beta1 con autoscaler y PDB derivados de los values; el API server real lo acepta (--dry-run=server)", r7)
    chk("R8 verify_flow: aprueba el flujo medido y suspende un flujo roto", lambda: all(verify_flow(REC["flow"]).values()) and not all(verify_flow({**REC["flow"], "loki_lines": 0, "traces_in_tempo": 290}).values()))
    ok = sum(v is True for v in R.values()); tot = sum(v is not None for v in R.values())
    for k, v in R.items():
        print(("✅" if v else ("⏭️ " if v is None else "❌")), k)
    print(f"\nPuntuación: {ok}/{tot} comprobaciones superadas" + ("" if (live and docker) else " (algunas se omiten sin helm/charts/Docker)"))
    return R

try:
    if LIVE_RENDER and DOCKER and API_LAB is None:
        API_LAB = K.ApiServerLab().__enter__()
        API_LAB.ensure_namespace("observability")
        op = API_LAB.helm_install("otel-operator", "otel-operator", [K8S / "operator" / "values-operator.yaml"], namespace="opentelemetry-operator-system", set_values={"admissionWebhooks.create": "false"})
    run_rubric()
except NameError as e:
    print("Define las funciones de los pasos primero:", e)
finally:
    if API_LAB is not None:
        API_LAB.__exit__(); API_LAB = None

## Retos extra (nivel experto)

1. **Despliegue real**: con kind/k3d en tu máquina, construye la imagen (`k8s/image/Dockerfile`), cárgala con `kind load docker-image`, instala los 7 releases en orden (Tempo/Loki/Prometheus → gateway → agente → Grafana → SupportOps) y **ejecuta `verify_flow` contra los backends del clúster** (`kubectl port-forward`).
2. **HPA por memoria de tail sampling**: escribe el `HorizontalPodAutoscaler` (v2, métrica `Pods`) sobre `otelcol_processor_tail_sampling_sampling_traces_on_memory` (con Prometheus Adapter o KEDA) y simula con `K.hpa_simulate` si cambia el comportamiento frente a CPU.
3. **Política de seguridad como código**: convierte `security_findings` en una regla de Kyverno o conftest y pruébala con `kubeconform`.
4. **Tres capas**: añade un Collector intermedio solo para `span_metrics`/`k8s_attributes` y mide con `K.affinity_experiment` qué cambia en la completitud de trazas.
5. **Upgrade sin pérdida**: simula un `helm upgrade` del gateway (mismo `decision_wait`) con 3 réplicas y calcula cuántas trazas en curso se reasignan (anillo de `load_balancing`).

## Reflexión

- ¿Qué pasaría con tu política de muestreo si el HPA duplicara las réplicas en pleno incidente (justo cuando más errores hay)? ¿Qué protege tu SLO de observabilidad?
- ¿Qué parte de este despliegue pondrías bajo GitOps y cuál mantendrías manual? ¿Por qué?
- ¿Qué validaciones de este proyecto **no** habrían detectado un `CrashLoopBackOff` por falta de memoria o una sonda mal puesta? ¿Cómo las cubrirías en un clúster real?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable: redefine las funciones y vuelve a lanzar la rúbrica.

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def gateway_overrides(decision_wait, num_traces, baseline_pct, replicas):
    return {"autoscaling": {"minReplicas": replicas},
            "config": {"processors": {"tail_sampling": {"decision_wait": decision_wait, "num_traces": num_traces,
                                                          "policies": [{"name": "errors", "type": "status_code", "status_code": {"status_codes": ["ERROR"]}},
                                                                       {"name": "slow", "type": "latency", "latency": {"threshold_ms": 5000}},
                                                                       {"name": "costly", "type": "ottl_condition", "ottl_condition": {"error_mode": "ignore", "span": ['attributes["supportops.cost_usd"] > 0.0005']}},
                                                                       {"name": "baseline", "type": "probabilistic", "probabilistic": {"sampling_percentage": baseline_pct}}]}}}}

def memory_budget(manifests, node_mem_mib, headroom=0.5):
    total = K.resource_requests(manifests)["memory_mib"]
    return total <= node_mem_mib * headroom, total

def security_findings(manifests):
    out = []
    for m in manifests:
        if m.get("kind") not in ("Deployment", "StatefulSet", "DaemonSet"):
            continue
        pod = m["spec"]["template"]["spec"]
        pod_nonroot = (pod.get("securityContext") or {}).get("runAsNonRoot") is True
        for c in pod.get("containers", []):
            sc = c.get("securityContext") or {}
            who = f"{m['kind']}/{m['metadata']['name']}/{c['name']}"
            if "memory" not in ((c.get("resources") or {}).get("limits") or {}):
                out.append(f"{who}: sin limits.memory")
            if not (pod_nonroot or sc.get("runAsNonRoot") is True):
                out.append(f"{who}: runAsNonRoot no está a true")
            if sc.get("readOnlyRootFilesystem") is not True:
                out.append(f"{who}: readOnlyRootFilesystem no está a true")
    return out

def validate_rendered(rendered):
    cfg = {"otel-gateway": ("collector", K.configmap_text(rendered["otel-gateway"], "otel-gateway", "relay")),
           "otel-agent": ("collector", K.configmap_text(rendered["otel-agent"], "otel-agent", "relay")),
           "tempo": ("tempo", K.configmap_text(rendered["tempo"], "tempo", "tempo.yaml")),
           "loki": ("loki", K.configmap_text(rendered["loki"], "loki", "config.yaml"))}
    return {k: K.validate_with_binary(kind, txt)[0] for k, (kind, txt) in cfg.items()}

AFFINITY_TYPES = {"latency", "status_code", "span_count", "numeric_attribute", "string_attribute", "boolean_attribute", "ottl_condition", "trace_state", "and", "composite"}
def needs_affinity(policies):
    return any(p["type"] in AFFINITY_TYPES for p in policies)

def desired_replicas(current, metric, target, min_r, max_r, tol=0.1):
    ratio = metric / target
    d = current if abs(ratio - 1) <= tol else math.ceil(current * ratio)
    return max(min_r, min(max_r, d))

def operator_cr(values, name="gateway", namespace="observability"):
    cfg = {k: v for k, v in values["config"].items() if k in ("receivers", "processors", "exporters", "connectors", "service")}
    cr = {"apiVersion": "opentelemetry.io/v1beta1", "kind": "OpenTelemetryCollector", "metadata": {"name": name, "namespace": namespace},
          "spec": {"mode": values["mode"], "image": f"{values['image']['repository']}:{values['image']['tag']}", "replicas": values["replicaCount"],
                   "resources": values["resources"], "config": cfg}}
    a = values.get("autoscaling") or {}
    if a.get("enabled"):
        cr["spec"]["autoscaler"] = {"minReplicas": a["minReplicas"], "maxReplicas": a["maxReplicas"], "targetCPUUtilization": a["targetCPUUtilizationPercentage"]}
    if (values.get("podDisruptionBudget") or {}).get("enabled"):
        cr["spec"]["podDisruptionBudget"] = {"minAvailable": values["podDisruptionBudget"]["minAvailable"]}
    return cr

def verify_flow(flow):
    frac = flow["traces_in_tempo"] / flow["tickets"]
    return {"trazas conservadas entre 10 % y 40 %": 0.10 <= frac <= 0.40, "hay logs en Loki": flow["loki_lines"] > 0, "hay métricas gen_ai en Prometheus": flow["prom_genai_series"] > 0,
            "hay span_metrics con llamadas": flow["span_metrics_calls"] > 0, "el gateway está up": len(flow["gateway_up"]) >= 1 and all(float(x) == 1.0 for x in flow["gateway_up"])}

if LIVE_RENDER and DOCKER:
    API_LAB = K.ApiServerLab().__enter__()
    API_LAB.ensure_namespace("observability")
    API_LAB.helm_install("otel-operator", "otel-operator", [K8S / "operator" / "values-operator.yaml"], namespace="opentelemetry-operator-system", set_values={"admissionWebhooks.create": "false"})
try:
    if LIVE_RENDER:
        print({k: len(security_findings(v)) for k, v in REND.items()}, "· memoria total (MiB):", round(memory_budget(sum(REND.values(), []), 8192)[1]))
    print(verify_flow(REC["flow"]))
    R = run_rubric()
finally:
    if API_LAB is not None:
        API_LAB.__exit__(); API_LAB = None